# Daily Data Pipeline — Fund Risk

This notebook implements the daily morning data validation and enrichment 
workflow that runs before any risk calculations begin. The automated 
pipeline has already loaded the previous day's positions into the database 
overnight. What follows is the validation, enrichment and exception review 
that must complete before risk metrics are computed and reported.


**Context**

Fund administrator position files arrive daily, typically between 7am and 
9am, reflecting the previous day's closing positions and prices. Before 
these feed into VaR models and stress tests, three questions must be 
answered:

1. **Are the prices correct?** Fund administrator pricing can differ from 
   Bloomberg, particularly for bonds and illiquid instruments where 
   model-based or stale prices are common.

2. **Are there new instruments?** Any ISIN that was not in yesterday's 
   portfolio requires reference data from Bloomberg before it can be 
   risk-managed correctly.

3. **Is the market context consistent with the risk numbers?** A VaR 
   figure means something different when the VIX is at 30 versus 15. 
   Market context is essential for interpreting risk metrics correctly.

The workflow below simulates this process for the **AIFM Hedge Fund** on 
**13 May 2026**, using a mock Bloomberg client that mirrors the real 
`blpapi` interface. Switching to production Bloomberg requires changing 
one import line.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.plot_style import ACCENT, ACCENT2, ACCENT3
import sqlalchemy as sa
from sqlalchemy import text
import warnings
warnings.filterwarnings('ignore')

# switch to real Bloomberg in one line
from src.mock_bloomberg import MockBloomberg as Bloomberg
from src.database import get_engine, query_positions, query_nav_history
from src.enrichment import enrich_positions, get_risk_ready_df

# ----------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------
FUND_ID    = 'AIFM_HedgeFund'
TODAY      = '2026-05-13'
YESTERDAY  = '2026-05-12'
ENGINE     = get_engine()
BBG        = Bloomberg()

print(f'Fund    : {FUND_ID}')
print(f'Date    : {TODAY}')
print(f'Engine  : {ENGINE.url}')

MockBloomberg: connected (simulation mode)
Swap import to RealBloomberg for production use.
Fund    : AIFM_HedgeFund
Date    : 2026-05-13
Engine  : sqlite:////Users/mrspatbile/Documents/coding/manco-risk-mngmt/data/risk_management.db


## 1. Load Today's Positions

Load the latest positions from the database and compare with yesterday
to identify what changed overnight: new instruments, removed instruments,
and significant position size changes.

In [2]:
# ----------------------------------------------------------------
# Ensure database and position files exist
# ----------------------------------------------------------------
from pathlib import Path
import subprocess

ROOT_DIR = Path('..').resolve()
DB_PATH  = ROOT_DIR / 'data' / 'risk_management.db'
DATA_DIR = ROOT_DIR / 'data'

if not DB_PATH.exists():
    print('Database not found. Generating position files and database...')
    subprocess.run(['python3', str(ROOT_DIR / 'src' / 'generate_positions.py')],
                   check=True)
    subprocess.run(['python3', str(ROOT_DIR / 'src' / 'database.py')],
                   check=True)
    print('Done.')
else:
    print(f'Database found: {DB_PATH}')
    print(f'Size: {DB_PATH.stat().st_size / 1024:.1f} KB')

Database found: /Users/mrspatbile/Documents/coding/manco-risk-mngmt/data/risk_management.db
Size: 1876.0 KB


In [3]:
# load today and yesterday
today     = query_positions(ENGINE, FUND_ID, TODAY)
yesterday = query_positions(ENGINE, FUND_ID, YESTERDAY)

print(f'Today    : {len(today)} positions')
print(f'Yesterday: {len(yesterday)} positions')

# what changed overnight?
new_isins     = set(today['isin']) - set(yesterday['isin'])
removed_isins = set(yesterday['isin']) - set(today['isin'])

print(f'\nNew instruments     : {len(new_isins)}')
print(f'Removed instruments : {len(removed_isins)}')

if new_isins:
    print(f'\nNew ISINs:')
    for isin in new_isins:
        name = today[today['isin'] == isin]['instrument_name'].values[0]
        print(f'  {isin} : {name}')

if removed_isins:
    print(f'\nRemoved ISINs:')
    for isin in removed_isins:
        name = yesterday[
            yesterday['isin'] == isin]['instrument_name'].values[0]
        print(f'  {isin} : {name}')

# portfolio snapshot
print(f'\n--- Portfolio snapshot ({TODAY}) ---')
nav = today['market_value_eur'].sum()
print(f'NAV                 : EUR {nav:,.0f}')
print(f'Long exposure       : EUR {today[today["market_value_eur"] > 0]["market_value_eur"].sum():,.0f}')
print(f'Short exposure      : EUR {today[today["market_value_eur"] < 0]["market_value_eur"].sum():,.0f}')
print(f'Gross exposure      : EUR {today["market_value_eur"].abs().sum():,.0f}')
print(f'\nAsset class breakdown:')
breakdown = today.groupby('asset_class')['market_value_eur'].sum().sort_values(ascending=False)
for ac, mv in breakdown.items():
    pct = mv / nav * 100
    print(f'  {ac:15s}: EUR {mv:>15,.0f}  ({pct:+.1f}%)')# load today and yesterday
today     = query_positions(ENGINE, FUND_ID, TODAY)
yesterday = query_positions(ENGINE, FUND_ID, YESTERDAY)

print(f'Today    : {len(today)} positions')
print(f'Yesterday: {len(yesterday)} positions')


Today    : 14 positions
Yesterday: 14 positions

New instruments     : 0
Removed instruments : 0

--- Portfolio snapshot (2026-05-13) ---
NAV                 : EUR 94,623,895
Long exposure       : EUR 105,310,570
Short exposure      : EUR -10,686,675
Gross exposure      : EUR 115,997,245

Asset class breakdown:
  Equity         : EUR      60,120,885  (+63.5%)
  FX             : EUR      15,922,100  (+16.8%)
  Cash           : EUR      10,000,000  (+10.6%)
  Bond           : EUR       8,983,190  (+9.5%)
  Derivative     : EUR        -402,280  (-0.4%)
Today    : 14 positions
Yesterday: 14 positions


## 2. Price Validation

Fund administrator prices can differ from Bloomberg, particularly for
bonds where model-based or matrix pricing is common, and for instruments
that have not traded recently. Any discrepancy above the tolerance
threshold requires investigation before risk metrics are computed.

Tolerance thresholds:
- **equities and FX**: 0.5%
- **bonds**: 0.25% — small price differences translate into significant
  P&L on large notionals once duration is applied

In [4]:
# pull Bloomberg prices for all liquid instruments
liquid = today[today['bloomberg_ticker'].notna()].copy()
tickers = liquid['bloomberg_ticker'].tolist()

bbg_prices = BBG.bdp(tickers, 'PX_LAST').reset_index()
bbg_prices = bbg_prices.rename(columns={
    'security': 'bloomberg_ticker',
    'PX_LAST' : 'bbg_price'
})

# merge fund admin price vs Bloomberg price
validation = liquid[['instrument_name', 'asset_class',
                      'bloomberg_ticker', 'price']].merge(
    bbg_prices, on='bloomberg_ticker', how='left'
)
validation = validation.rename(columns={'price': 'fund_admin_price'})

# compute discrepancy
validation['diff_pct'] = (
    (validation['fund_admin_price'] - validation['bbg_price']) /
    validation['bbg_price'] * 100
).round(4)

# flag by asset class tolerance
def flag_discrepancy(row):
    if pd.isna(row['bbg_price']):
        if row['asset_class'] == 'Derivative':
            return 'OTC - valuation agent'
        return 'NO BBG PRICE'
    tolerance = 0.25 if row['asset_class'] == 'Bond' else 0.50
    if abs(row['diff_pct']) > tolerance:
        return f'FLAG (>{tolerance}%)'
    return 'OK'

validation['status'] = validation.apply(flag_discrepancy, axis=1)

# display
print('--- Price validation report ---')
print(f'{"Instrument":<30} {"Asset Class":<12} {"Fund Admin":>12} {"Bloomberg":>12} {"Diff%":>8} {"Status"}')
print('-' * 90)
for _, row in validation.iterrows():
    bbg_str = f'{row["bbg_price"]:>12.4f}' if not pd.isna(
        row['bbg_price']) else f'{"N/A":>12}'
    print(f'{row["instrument_name"]:<30} {row["asset_class"]:<12} '
          f'{row["fund_admin_price"]:>12.4f} {bbg_str} '
          f'{row["diff_pct"]:>8.4f} {row["status"]}')

flagged = validation[validation['status'].str.startswith('FLAG')]
print(f'\nFlagged: {len(flagged)} / {len(validation)} positions')
if len(flagged) > 0:
    print('Requires investigation before risk metrics are computed.')

--- Price validation report ---
Instrument                     Asset Class    Fund Admin    Bloomberg    Diff% Status
------------------------------------------------------------------------------------------
SPDR S&P 500 ETF               Equity           523.4200     523.4200   0.0000 OK
Apple Inc                      Equity           211.4500     211.4500   0.0000 OK
Microsoft Corp                 Equity           415.3200     415.3200   0.0000 OK
JPMorgan Chase                 Equity           248.7300     248.7300   0.0000 OK
Euro Stoxx 50 Future           Equity          5124.8700    5124.8700   0.0000 OK
Tesla Inc (Short)              Equity           175.3400     175.3400   0.0000 OK
Nvidia Corp (Short)            Equity           892.5400     892.5400   0.0000 OK
T 2.875 05/15/28               Bond              96.4200      96.4200   0.0000 OK
DBR 0 08/15/29                 Bond              90.8700      90.8700   0.0000 OK
LVMH 3.5 06/15/31              Bond              98.3

In this simulation all prices match since both the fund administrator
and Bloomberg use the same mock data source. In production,
discrepancies appear across instrument types as follows:

**Fixed income**

- **Liquid bonds** (on-the-run government bonds, large IG corporates):
  actively traded, Bloomberg shows real transaction prices. Discrepancies
  are rare and typically small.
- **Illiquid bonds** (small IG corporates, HY, private placements):
  rarely traded. Bloomberg applies matrix pricing, interpolating from
  comparable instruments with similar issuer, maturity, and rating.
  The fund administrator may use a different matrix or reference date,
  producing discrepancies that require investigation.
- **Private loans and CLOs**: no market price exists. Both Bloomberg
  and the fund administrator use discounted cash flow models with an
  assumed discount rate. Different valuation agents apply different
  rate assumptions, so discrepancies here are expected and larger.

**Derivatives**

- **Listed options and futures**: exchange prices available on Bloomberg
  using the full contract ticker (e.g. `SPXW 260619P05500 Index`).
  Validated the same way as equities.
- **OTC derivatives** (swaps, bespoke structures): no Bloomberg market
  price. Validated against counterparty marks or an independent
  valuation agent. Discrepancies trigger a formal dispute process.

---

## 3. New Instrument Onboarding

When a new ISIN appears in today's positions that was not in yesterday's
portfolio, its reference data must be pulled from Bloomberg and validated
before it enters the risk system. Two checks are critical:

- **asset class consistency**: does Bloomberg classify the instrument
  the same way as the fund administrator?
- **duration sanity**: for bonds, duration must be lower than maturity.
  A significant deviation signals a data error or a misclassified
  instrument.

In this simulation no new instruments appeared overnight. The cells
below show the onboarding workflow that would run if new ISINs were
detected.

In [5]:
# ----------------------------------------------------------------
# New instrument onboarding
# ----------------------------------------------------------------

if new_isins:
    print(f'Onboarding {len(new_isins)} new instruments...\n')
    for isin in new_isins:
        row    = today[today['isin'] == isin].iloc[0]
        ticker = row['bloomberg_ticker']

        if pd.isna(ticker):
            print(f'{isin}: no Bloomberg ticker - illiquid instrument')
            print(f'  fund admin asset class: {row["asset_class"]}')
            print(f'  manual onboarding required\n')
            continue

        # pull reference data from Bloomberg
        ref = BBG.bdp(ticker, [
            'NAME', 'ASSET_CLASS', 'CRNCY',
            'DUR_ADJ_MID', 'CONVEXITY', 'YLD_YTM_MID',
            'BETA', 'RTG_SP', 'VOLUME_AVG_20D'
        ])

        bbg_asset_class = ref.loc[ticker, 'ASSET_CLASS']
        bbg_duration    = ref.loc[ticker, 'DUR_ADJ_MID']

        print(f'New instrument: {row["instrument_name"]}')
        print(f'  ISIN              : {isin}')
        print(f'  Bloomberg ticker  : {ticker}')
        print(f'  Fund admin class  : {row["asset_class"]}')
        print(f'  Bloomberg class   : {bbg_asset_class}')

        # asset class consistency check
        if str(bbg_asset_class) != str(row['asset_class']):
            print(f'  *** MISMATCH: asset class conflict - '
                  f'investigate before adding to risk system ***')
        else:
            print(f'  Asset class       : OK')

        # duration sanity check for bonds
        if row['asset_class'] == 'Bond' and not pd.isna(bbg_duration):
            maturity_years = (
                pd.Timestamp(row['maturity']) -
                pd.Timestamp(TODAY)
            ).days / 365 if row['maturity'] else None

            if maturity_years and bbg_duration > maturity_years:
                print(f'  *** DURATION ERROR: duration {bbg_duration:.2f} '
                      f'exceeds maturity {maturity_years:.2f}y ***')
            else:
                print(f'  Duration          : {bbg_duration:.2f} years OK')

        print()

else:
    print('No new instruments today.')
    print()
    print('Onboarding workflow (runs when new ISINs detected):')
    print('  1. pull reference data from Bloomberg (bdp)')
    print('  2. validate asset class vs fund administrator')
    print('  3. validate duration vs maturity for bonds')
    print('  4. flag mismatches for manual investigation')
    print('  5. add to risk system only after validation passes')

No new instruments today.

Onboarding workflow (runs when new ISINs detected):
  1. pull reference data from Bloomberg (bdp)
  2. validate asset class vs fund administrator
  3. validate duration vs maturity for bonds
  4. flag mismatches for manual investigation
  5. add to risk system only after validation passes


## 4. Market Context

Before reviewing risk metrics, current market conditions must be
assessed. A VaR figure or stress test result is only meaningful in
context: the same number carries a different implication when credit
spreads are widening, rates are moving sharply, or implied volatility
is elevated.

The three key inputs pulled from Bloomberg each morning:

- **VIX**: options market implied volatility on the S&P 500, the
  primary fear gauge. Levels above 25 signal elevated stress.
- **Yield curve**: overnight moves in government bond yields directly
  affect the duration P&L of the fixed income portfolio.
- **Credit spreads**: z-spreads on corporate bond holdings indicate
  whether credit conditions are tightening or easing.

In [6]:
# ----------------------------------------------------------------
# Market context: VIX, yields, credit spreads
# ----------------------------------------------------------------

# VIX: fear gauge
vix = BBG.bdp('VIX Index', 'PX_LAST')
vix_level = vix.loc['VIX Index', 'PX_LAST']

vix_regime = (
    'ELEVATED (>25): heightened stress'   if vix_level > 25 else
    'MODERATE (15-25): normal conditions' if vix_level > 15 else
    'LOW (<15): complacent market'
)

print(f'--- VIX ---')
print(f'Current level : {vix_level:.2f}')
print(f'Regime        : {vix_regime}')

# yield curve: overnight moves on bonds in portfolio
print(f'\n--- Yield curve (overnight moves) ---')
bond_tickers = today[
    (today['asset_class'] == 'Bond') &
    (today['bloomberg_ticker'].notna())
]['bloomberg_ticker'].tolist()

if bond_tickers:
    yields_today     = BBG.bdh(bond_tickers, 'YLD_YTM_MID',
                               TODAY, TODAY)
    yields_yesterday = BBG.bdh(bond_tickers, 'YLD_YTM_MID',
                               YESTERDAY, YESTERDAY)

    for ticker in bond_tickers:
        name = today[
            today['bloomberg_ticker'] == ticker
        ]['instrument_name'].values[0]
        try:
            ytm_today = yields_today.loc[
                TODAY, 'YLD_YTM_MID'] if len(bond_tickers) == 1 \
                else yields_today.xs(ticker, level='security').iloc[-1]['YLD_YTM_MID']
            ytm_yest  = yields_yesterday.loc[
                YESTERDAY, 'YLD_YTM_MID'] if len(bond_tickers) == 1 \
                else yields_yesterday.xs(ticker, level='security').iloc[-1]['YLD_YTM_MID']
            move_bps  = (ytm_today - ytm_yest) * 100
            print(f'  {name:<25}: {ytm_today:.3f}% '
                  f'({move_bps:+.1f}bps overnight)')
        except Exception:
            print(f'  {name:<25}: yield data unavailable')

# credit spreads on corporate bonds in portfolio
print(f'\n--- Credit spreads ---')
corp_tickers = today[
    (today['sub_asset_class'].isin(['IG Corporate', 'HY Corporate'])) &
    (today['bloomberg_ticker'].notna())
]['bloomberg_ticker'].tolist()

if corp_tickers:
    spreads = BBG.bdp(corp_tickers, ['Z_SPRD_MID', 'RTG_SP'])
    for ticker in corp_tickers:
        name = today[
            today['bloomberg_ticker'] == ticker
        ]['instrument_name'].values[0]
        z_sprd = spreads.loc[ticker, 'Z_SPRD_MID']
        rating = spreads.loc[ticker, 'RTG_SP']
        if not pd.isna(z_sprd):
            print(f'  {name:<25}: z-spread {z_sprd:.0f}bps '
                  f'(rating: {rating})')
else:
    print('  No corporate bonds in portfolio.')

print(f'\n--- Summary ---')
print(f'VIX at {vix_level:.1f}: {vix_regime}')
print(f'Review risk metrics in this context before reporting.')

--- VIX ---
Current level : 18.42
Regime        : MODERATE (15-25): normal conditions

--- Yield curve (overnight moves) ---
  T 2.875 05/15/28         : 4.420% (+0.0bps overnight)
  DBR 0 08/15/29           : 2.310% (+0.0bps overnight)
  LVMH 3.5 06/15/31        : 3.890% (+0.0bps overnight)

--- Credit spreads ---
  LVMH 3.5 06/15/31        : z-spread 58bps (rating: A+)

--- Summary ---
VIX at 18.4: MODERATE (15-25): normal conditions
Review risk metrics in this context before reporting.


## 5. Exception Review

Review automated flags from overnight processing before risk metrics
are computed and reported. Three categories of exceptions:

- **price outliers**: positions where the overnight price move exceeds
  2 standard deviations of the 20-day rolling volatility
- **missing sensitivities**: liquid instruments without Bloomberg
  enrichment data needed for stress testing
- **limit proximity**: positions approaching VaR or concentration limits

In [7]:
# ----------------------------------------------------------------
# Exception review
# ----------------------------------------------------------------

# --- 1. Price outliers ---
print('--- 1. Price outliers (>2 sigma overnight move) ---')

# get 20 days of price history for liquid instruments
liquid_tickers = today[
    today['bloomberg_ticker'].notna()
]['bloomberg_ticker'].tolist()

outliers = []
for ticker in liquid_tickers:
    hist = BBG.bdh(ticker, 'PX_LAST', '2026-04-01', TODAY)
    if len(hist) < 5:
        continue

    returns  = hist['PX_LAST'].pct_change().dropna()
    vol_20d  = returns.rolling(20).std().iloc[-1]
    ret_1d   = returns.iloc[-1]

    if vol_20d > 0 and abs(ret_1d) > 2 * vol_20d:
        name = today[
            today['bloomberg_ticker'] == ticker
        ]['instrument_name'].values[0]
        outliers.append({
            'instrument': name,
            'return_1d' : ret_1d * 100,
            'vol_20d'   : vol_20d * 100,
            'z_score'   : ret_1d / vol_20d,
        })

if outliers:
    for o in outliers:
        print(f'  {o["instrument"]:<30}: '
              f'1d return {o["return_1d"]:+.2f}%, '
              f'20d vol {o["vol_20d"]:.2f}%, '
              f'z-score {o["z_score"]:+.2f}')
else:
    print('  No price outliers detected.')

# --- 2. Missing sensitivities ---
print('\n--- 2. Missing sensitivities ---')

missing = []
for _, row in today.iterrows():
    if pd.isna(row['bloomberg_ticker']):
        continue  # illiquid, expected to have no Bloomberg data

    issues = []
    if row['asset_class'] == 'Bond':
        if pd.isna(row.get('dur_adj_mid')):
            issues.append('duration missing')
        if pd.isna(row.get('convexity')):
            issues.append('convexity missing')
    elif row['asset_class'] == 'Equity':
        if pd.isna(row.get('beta')):
            issues.append('beta missing')

    if issues:
        missing.append({
            'instrument': row['instrument_name'],
            'issues'    : ', '.join(issues)
        })

if missing:
    for m in missing:
        print(f'  {m["instrument"]:<30}: {m["issues"]}')
else:
    print('  All liquid instruments have required sensitivities.')

# --- 3. Concentration check ---
print('\n--- 3. Concentration check (single position > 10% NAV) ---')

nav        = today['market_value_eur'].sum()
large_pos  = today[today['market_value_eur'].abs() / abs(nav) > 0.10]

if len(large_pos) > 0:
    for _, row in large_pos.iterrows():
        pct = row['market_value_eur'] / nav * 100
        print(f'  {row["instrument_name"]:<30}: '
              f'{pct:+.1f}% of NAV - review concentration limit')
else:
    print('  No concentration limit breaches detected.')

print('\n--- Exception review complete ---')
print('Proceed to risk calculations.' if not outliers and
      not missing else
      'Resolve flagged exceptions before reporting.')

--- 1. Price outliers (>2 sigma overnight move) ---
  SPDR S&P 500 ETF              : 1d return +3.86%, 20d vol 1.62%, z-score +2.38
  Tesla Inc (Short)             : 1d return +3.18%, 20d vol 1.59%, z-score +2.00

--- 2. Missing sensitivities ---
  SPDR S&P 500 ETF              : beta missing
  Apple Inc                     : beta missing
  Microsoft Corp                : beta missing
  JPMorgan Chase                : beta missing
  Euro Stoxx 50 Future          : beta missing
  Tesla Inc (Short)             : beta missing
  Nvidia Corp (Short)           : beta missing
  T 2.875 05/15/28              : duration missing, convexity missing
  DBR 0 08/15/29                : duration missing, convexity missing
  LVMH 3.5 06/15/31             : duration missing, convexity missing

--- 3. Concentration check (single position > 10% NAV) ---
  SPDR S&P 500 ETF              : +24.6% of NAV - review concentration limit
  Apple Inc                     : +15.9% of NAV - review concentration limit

**Note on missing sensitivities**: the check above runs on raw
positions before Bloomberg enrichment. Section 6 will enrich all
liquid instruments with Bloomberg sensitivities. Any remaining
missing values after enrichment represent genuine data gaps
requiring manual intervention.

## 6. Enrich Positions and Produce Risk-Ready DataFrame

With price validation and exception review complete, positions are
enriched with Bloomberg sensitivities. Liquid instruments pull
reference data via `bdp`. Illiquid instruments (direct properties,
private loans) use sensitivities already present from the fund
administrator Excel.

The output is a single risk-ready DataFrame containing everything
needed for VaR, stress testing, and liquidity calculations in the
domain risk notebooks.

In [8]:
# ----------------------------------------------------------------
# Enrich positions and produce risk-ready DataFrame
# ----------------------------------------------------------------

from src.enrichment import enrich_positions, get_risk_ready_df

print(f'Enriching positions for {FUND_ID} on {TODAY}...\n')

enriched = enrich_positions(ENGINE, FUND_ID, TODAY, BBG)

# risk-ready DataFrame
risk_df = get_risk_ready_df(ENGINE, FUND_ID, TODAY)

print(f'\n--- Risk-ready DataFrame ---')
print(f'Rows     : {len(risk_df)}')
print(f'Columns  : {len(risk_df.columns)}')
print(f'\nEnrichment summary:')
print(risk_df.groupby('enrichment_source')[
    ['instrument_name']].count().rename(
    columns={'instrument_name': 'positions'}))

print(f'\n--- Sensitivities coverage ---')
print(f'Beta present     : {risk_df["beta"].notna().sum()} / {len(risk_df)}')
print(f'Duration present : {risk_df["dur_adj_mid"].notna().sum()} / {len(risk_df)}')
print(f'ADV present      : {(risk_df["adv_eur"] > 0).sum()} / {len(risk_df)}')

print(f'\n--- Top 10 positions (risk-ready) ---')
display_cols = ['instrument_name', 'asset_class',
                'market_value_eur', 'weight_pct',
                'beta', 'dur_adj_mid', 'enrichment_source']
print(risk_df[display_cols].to_string(index=False))

Enriching positions for AIFM_HedgeFund on 2026-05-13...

Enriching 14 positions for AIFM_HedgeFund on 2026-05-13...
  bloomberg enriched : 13 positions
  fund admin         : 1 positions
  total              : 14 positions

--- Risk-ready DataFrame ---
Rows     : 14
Columns  : 29

Enrichment summary:
                   positions
enrichment_source           
bloomberg                 13
none                       1

--- Sensitivities coverage ---
Beta present     : 7 / 14
Duration present : 3 / 14
ADV present      : 10 / 14

--- Top 10 positions (risk-ready) ---
     instrument_name asset_class  market_value_eur  weight_pct  beta  dur_adj_mid enrichment_source
    SPDR S&P 500 ETF      Equity        23292190.0     24.6155  1.00          NaN         bloomberg
      Microsoft Corp      Equity        22178088.0     23.4381  0.91          NaN         bloomberg
           Apple Inc      Equity        15055240.0     15.9106  1.24          NaN         bloomberg
            Cash EUR        Cash

## Summary

The daily data pipeline is complete. The risk-ready DataFrame contains:

- **14 positions** enriched from two sources: Bloomberg (13) and
  fund administrator (1 cash position with no Bloomberg ticker)
- **beta** available for all equity positions
- **duration and convexity** available for all bond positions
- **ADV** available for all liquid instruments for liquidity calculations
- **audit trail**: `enrichment_source` column documents the data
  source for every sensitivity, available for regulatory review

This DataFrame is the input to all downstream risk calculations:
VaR, Expected Shortfall, Annex VI stress scenarios, and ESMA
liquidity stress testing.